In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from apps.onpe.schemas import ConsultaElectoral, Eleccion
from typing import Literal
from apps.onpe.client import get_resultados_onpe, cookies, headers
from apps.onpe.enums import UbigeoNivel1, UbigeoNivel2, UbigeoNivel3, AmbitoGeografico
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ThreadPoolExecutor
import requests

In [3]:
def get_resultados_onpe(consulta_electoral: ConsultaElectoral, mode: Literal['resumen', 'detalle', 'participacion']):
    params = consulta_electoral.get_params(mode)
    response = requests.get(
        consulta_electoral.get_resultados_url() if mode == 'detalle' else consulta_electoral.get_resumen_url() if mode == 'resumen' else consulta_electoral.get_participacion_url(),
        params=params,
        headers=headers,
        cookies=cookies,
    )
    return response.json()

In [4]:
consulta_electoral = ConsultaElectoral(eleccion=Eleccion.PRESIDENCIAL)

params = {
    "tipoFiltro": "total",
}

response = requests.get(
    consulta_electoral.get_participacion_url(),
    params=params,
    headers=headers,
    cookies=cookies,
)
data_participacion = response.json()['data']
data_participacion

{'totalElectoresHabiles': 27325432,
 'totalAsistentes': 18818432,
 'totalAusentes': 6461466,
 'porcentajeAsistentes': 68.868,
 'porcentajeAusentes': 23.646,
 'ubigeoNivel01': None,
 'ubigeoNivel02': None,
 'ubigeoNivel03': None,
 'idLocalVotacion': None}

In [5]:
data_participacion['totalAsistentes'] / (data_participacion['totalAusentes'] + data_participacion['totalAsistentes'])

0.7444030035247768

In [50]:
EX_ELECTORES_HABILES = 1_210_813
EX_AUSENTES = 544_307
EX_ASISTENTES = 275_290
EX_NORMAL_CONSTANT = EX_ELECTORES_HABILES / (EX_AUSENTES + EX_ASISTENTES)
EX_PRED_AUSENTES = EX_AUSENTES * EX_NORMAL_CONSTANT
EX_PRED_ASISTENTES = EX_ASISTENTES * EX_NORMAL_CONSTANT

PERU_ELECTORES_HABILES = 26_114_619
PERU_AUSENTES = 5_917_159
PERU_ASISTENTES = 18_543_622
PERU_NORMAL_CONSTANT = PERU_ELECTORES_HABILES / (PERU_AUSENTES + PERU_ASISTENTES)
PERU_PRED_AUSENTES = PERU_AUSENTES * PERU_NORMAL_CONSTANT
PERU_PRED_ASISTENTES = PERU_ASISTENTES * PERU_NORMAL_CONSTANT

TOTAL_AUSENTES = EX_PRED_AUSENTES + PERU_PRED_AUSENTES
TOTAL_ASISTENTES = EX_PRED_ASISTENTES + PERU_PRED_ASISTENTES

In [51]:
int(EX_PRED_ASISTENTES), int(PERU_PRED_ASISTENTES)

(406693, 19797390)

In [52]:
18_543_622 + 275_290

18818912

In [39]:
18_818_912 * 1.0808897831047475 / 27_325_432

0.7444043230477502

In [29]:
18_818_912

18818912

In [28]:
TOTAL_ASISTENTES

20203959.549588975

In [52]:
asistentes = data_participacion['totalAsistentes'] / data_participacion['totalElectoresHabiles']
ausentes = data_participacion['totalAusentes'] / data_participacion['totalElectoresHabiles']

turnout = asistentes / (asistentes + ausentes)
turnout

0.7485430634176156

In [43]:
asistentes = data_participacion['totalAsistentes'] / data_participacion['totalElectoresHabiles']
ausentes = data_participacion['totalAusentes'] / data_participacion['totalElectoresHabiles']

turnout = asistentes / (asistentes + ausentes)
turnout

0.31606387421416626

In [38]:
ASISTENTES = 67.941
AUSENTES = 22.820

TURNOUT = ASISTENTES / (ASISTENTES + AUSENTES)